# Intermediate 06 — Multivariate Distributions and the Multivariate Normal

Practice notebook for the **"Multivariate Distributions and the Multivariate Normal"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Joint, marginal, and conditional distributions

For a random vector \( X = (X_1,\dots,X_d)^\top \), the joint distribution
is described by a joint pdf \( f_{X}(x) \) (continuous) or pmf (discrete).

The **marginal** of \( X_1 \) integrates out the other coordinates:

$$
f_{X_1}(x_1) = \int_{\mathbb{R}^{d-1}} f_{X}(x_1,x_2,\dots,x_d)\,dx_2\cdots dx_d.
$$

The **conditional** density of \( X_1 \) given \( X_2=x_2,\dots,X_d=x_d \) is

$$
f_{X_1 \mid X_2,\dots,X_d}(x_1 \mid x_2,\dots,x_d)
= \frac{f_{X}(x_1,\dots,x_d)}{f_{X_2,\dots,X_d}(x_2,\dots,x_d)},
$$

whenever the denominator is positive.

To **sample** from a multivariate normal \( N_d(\mu,\Sigma) \) we use the
Cholesky factor \( L \) with \( \Sigma = LL^\top \): if \( Z\sim N_d(0,I) \),
then \( X = \mu + LZ \sim N_d(\mu,\Sigma) \). Below we draw a
bivariate sample and check that the empirical covariance matches \( \Sigma \).


In [ ]:
# Sample bivariate normal via Cholesky and verify the sample covariance
rng = np.random.default_rng(42)

mu = np.array([1.0, -0.5])
rho = 0.6
sig1, sig2 = 2.0, 1.5
Sigma = np.array([[sig1**2, rho*sig1*sig2],
                  [rho*sig1*sig2, sig2**2]])

# Cholesky factor: Sigma = L L^T
L = np.linalg.cholesky(Sigma)

N = 200_000
Z = rng.standard_normal(size=(N, 2))
X = mu + Z @ L.T          # each row is a sample

sample_mean = X.mean(axis=0)
sample_cov  = np.cov(X, rowvar=False, ddof=1)

print("target mean  :", mu)
print("sample mean  :", sample_mean)
print()
print("target Sigma :\n", Sigma)
print("sample cov   :\n", sample_cov)
print()
print("max |Sigma - sample_cov| =", np.max(np.abs(Sigma - sample_cov)))


**Your turn:** change \( \rho \) to \( -0.9 \) and re-run. Does the sample
covariance still match? What happens to the shape of the cloud?


---
## Part 2 — The covariance matrix and portfolio variance

For \( X = (X_1,\dots,X_d)^\top \) with mean vector \( \mu=E[X] \),
the **covariance matrix** is

$$
\Sigma = \operatorname{Cov}(X) = E\big[(X-\mu)(X-\mu)^\top\big].
$$

The \( (i,j) \) entry is \( \operatorname{Cov}(X_i,X_j) \); \( \Sigma \) is symmetric
and positive semi-definite. For a portfolio \( R_p = w^\top R \),

$$
\operatorname{Var}(R_p) = w^\top \Sigma w.
$$

We verify this formula by comparing \( w^\top\Sigmaw \) against the
sample variance of \( R_p = w^\top X \) computed from the simulated draws.


In [ ]:
# Portfolio variance: w^T Sigma w vs sample variance of w^T X
rng = np.random.default_rng(42)

# 3-asset returns
mu = np.array([0.05, 0.08, 0.04])
Sig = np.array([[0.10, 0.02, 0.01],
                [0.02, 0.16, -0.03],
                [0.01, -0.03, 0.09]])
L = np.linalg.cholesky(Sig)

N = 300_000
Z = rng.standard_normal(size=(N, 3))
R = mu + Z @ L.T

w = np.array([0.4, 0.3, 0.3])          # portfolio weights (sum to 1)
w = w / w.sum()

Rp = R @ w                            # portfolio returns
var_formula = w @ Sig @ w
var_sample  = Rp.var(ddof=1)

print("weights          :", w)
print("w^T Sigma w      :", var_formula)
print("sample Var(R_p)  :", var_sample)
print("relative error   :", abs(var_formula - var_sample) / var_formula)

# Check positive semi-definiteness: eigenvalues of Sigma
eig = np.linalg.eigvalsh(Sig)
print("eigenvalues of Sigma:", eig, " (all >= 0?)", np.all(eig >= -1e-12))


**Your turn:** find weights \( w \) (not all zero) that minimize
\( w^\top\Sigmaw \) subject to \( 1^\topw=1 \). Hint: the
minimum-variance portfolio is proportional to \( \Sigma^{-1}1 \).


---
## Part 3 — The multivariate normal density and its contours

A random vector \( X\in\mathbb{R}^d \) is **multivariate normal** with mean
\( \mu \) and covariance \( \Sigma \) (positive definite), written
\( X\sim N_d(\mu,\Sigma) \), if its pdf is

$$
f_{X}(x) =
\frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}
\exp\left(-\tfrac{1}{2}(x-\mu)^\top \Sigma^{-1}(x-\mu)\right).
$$

The level sets \( \{x:(x-\mu)^\top\Sigma^{-1}(x-\mu)=c\} \)
are ellipses (in 2D) whose axes are aligned with the eigenvectors of \( \Sigma \).
We plot the density contours and overlay a sample cloud.


In [ ]:
# MVN density contours via the analytic pdf, plus a sample cloud
rng = np.random.default_rng(42)

mu = np.array([0.0, 0.0])
Sigma = np.array([[2.0, 0.9],
                  [0.9, 1.0]])
L = np.linalg.cholesky(Sigma)

# Grid
gx = np.linspace(-4, 4, 200)
gy = np.linspace(-3, 3, 200)
XX, YY = np.meshgrid(gx, gy)
pts = np.stack([XX.ravel(), YY.ravel()], axis=1)   # (M, 2)

# Analytic MVN pdf
invS = np.linalg.inv(Sigma)
dM = pts - mu
quad = np.einsum('ij,jk,ik->i', dM, invS, dM)
detS = np.linalg.det(Sigma)
pdf = np.exp(-0.5 * quad) / (2 * np.pi * np.sqrt(detS))
pdf = pdf.reshape(XX.shape)

# Samples
N = 5_000
Z = rng.standard_normal(size=(N, 2))
Xs = mu + Z @ L.T

plt.figure(figsize=(8, 6))
plt.contour(XX, YY, pdf, levels=10, colors="steelblue", linewidths=1.2)
plt.scatter(Xs[:,0], Xs[:,1], s=6, alpha=0.25, color="firebrick", label="samples")
plt.plot(*mu, "k^", ms=10, label="mean")
plt.axis("equal"); plt.xlabel("x1"); plt.ylabel("x2")
plt.title("Bivariate normal density contours vs samples")
plt.legend(); plt.tight_layout(); plt.show()


**Your turn:** rotate \( \Sigma \) by changing the off-diagonal entry from
\( 0.9 \) to \( -0.9 \). How do the contour ellipses tilt?


---
## Part 4 — Conditional distributions of the MVN

For the bivariate normal \( (X_1,X_2)^\top \sim N_2(\mu,\Sigma) \) with

$$
\mu = \begin{pmatrix}\mu_1 \\ \mu_2\end{pmatrix},\quad
\Sigma = \begin{pmatrix}\sigma_1^2 & \rho\sigma_1\sigma_2 \\
\rho\sigma_1\sigma_2 & \sigma_2^2\end{pmatrix},
$$

the conditional distribution of \( X_2 \) given \( X_1=x \) is normal with

$$
E[X_2 \mid X_1=x] = \mu_2 + \rho\frac{\sigma_2}{\sigma_1}(x-\mu_1),
\qquad
\operatorname{Var}(X_2 \mid X_1=x) = \sigma_2^2(1-\rho^2).
$$

The conditional mean is a **linear** function of \( x \) — the probabilistic origin
of simple linear regression. We verify both formulas by simulation: bin samples
near \( X_1=x \) and compare their mean/variance to the formulas.


In [ ]:
# Verify conditional mean & variance formulas by simulation
rng = np.random.default_rng(42)

mu1, mu2 = 0.0, 1.0
s1, s2 = 1.5, 1.0
rho = 0.7
Sigma = np.array([[s1**2, rho*s1*s2],
                  [rho*s1*s2, s2**2]])
L = np.linalg.cholesky(Sigma)

N = 400_000
Z = rng.standard_normal(size=(N, 2))
X = mu + np.zeros((N,2))  # placeholder
# use explicit mu vector
mu = np.array([mu1, mu2])
X = mu + Z @ L.T
X1, X2 = X[:,0], X[:,1]

# Conditional mean / var as functions of x
def cond_mean(x):  return mu2 + rho*(s2/s1)*(x - mu1)
def cond_var(x):    return s2**2 * (1 - rho**2)

# Pick several x values, bin samples in a window, compare
x_grid = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 0.15  # half-window width

print(f"{'x':>6} {'E[X2|X1=x] formula':>20} {'bin mean':>12} "
      f"{'Var formula':>14} {'bin var':>12} {'n':>7}")
for x in x_grid:
    mask = np.abs(X1 - x) < h
    bin_mean = X2[mask].mean()
    bin_var  = X2[mask].var(ddof=1)
    print(f"{x:6.2f} {cond_mean(x):20.5f} {bin_mean:12.5f} "
          f"{cond_var(x):14.5f} {bin_var:12.5f} {mask.sum():7d}")

# Plot conditional mean line over the scatter
plt.figure(figsize=(8,5))
plt.scatter(X1[:5000], X2[:5000], s=5, alpha=0.2, label="samples")
xs = np.linspace(X1.min(), X1.max(), 100)
plt.plot(xs, cond_mean(xs), "r-", lw=2, label=r"$E[X_2\mid X_1=x]$")
# +/- 1 cond std band
cs = np.sqrt(cond_var(xs))
plt.fill_between(xs, cond_mean(xs)-cs, cond_mean(xs)+cs, color="red", alpha=0.15)
plt.xlabel("x1"); plt.ylabel("x2"); plt.legend()
plt.title(r"Conditional mean and +/-1 sd band for $X_2\mid X_1=x$")
plt.tight_layout(); plt.show()


**Your turn:** derive the conditional distribution of \( X_1 \mid X_2=y \) by
swapping indices. Confirm by simulation that the slope is
\( \rho\sigma_1/\sigma_2 \) and the conditional variance is \( \sigma_1^2(1-\rho^2) \).


---
## Part 5 — Linear combinations of an MVN are normal

A defining property of the multivariate normal: for any fixed vector
\( a\in\mathbb{R}^d \), the linear combination \( a^\topX \) is
univariate normal with

$$
a^\topX \sim N\big(a^\top\mu,\ a^\top\Sigmaa\big).
$$

We verify this with a Kolmogorov–Smirnov test: the empirical distribution of
\( a^\topX \) should match \( N(a^\top\mu,a^\top\Sigmaa) \).
The same property implies any sub-vector of \( X \) is again multivariate normal.


In [ ]:
# Linear combinations of MVN are normal: KS check + histogram overlay
rng = np.random.default_rng(42)

mu = np.array([0.5, -1.0, 2.0])
Sigma = np.array([[1.0, 0.3, -0.2],
                  [0.3, 2.0, 0.5],
                  [-0.2, 0.5, 0.8]])
L = np.linalg.cholesky(Sigma)

N = 200_000
Z = rng.standard_normal(size=(N, 3))
X = mu + Z @ L.T

a = np.array([0.7, -1.2, 0.4])
Y = X @ a                       # a^T X

mean_th = a @ mu
var_th  = a @ Sigma @ a
print(f"a^T mu = {mean_th:.5f}   sample mean of Y = {Y.mean():.5f}")
print(f"a^T Sigma a = {var_th:.5f}   sample var of Y = {Y.var(ddof=1):.5f}")

# KS test against the theoretical normal
D, p = stats.kstest((Y - mean_th)/np.sqrt(var_th), "norm")
print(f"KS statistic = {D:.5f}   p-value = {p:.4f}  (>0.05 -> consistent with normal)")

# Histogram vs theoretical pdf
plt.figure(figsize=(8,5))
plt.hist(Y, bins=80, density=True, alpha=0.5, color="steelblue", label="sample")
ys = np.linspace(Y.min(), Y.max(), 200)
plt.plot(ys, stats.norm.pdf(ys, mean_th, np.sqrt(var_th)), "r-", lw=2, label="theory")
plt.xlabel("y"); plt.ylabel("density"); plt.legend()
plt.title(r"$a^\top X$ is normal with predicted mean/variance")
plt.tight_layout(); plt.show()


**Your turn:** pick another \( a \) and re-run. Try \( a=(1,0,0) \)
— does \( X_1 \) alone match \( N(\mu_1,\sigma_{11}) \)?


---
## Wrap-up

You have now verified, by simulation, the four pillars of the multivariate normal:

1. **Sampling** via the Cholesky factor reproduces \( \Sigma \) in the sample covariance.
2. **Portfolio variance** \( w^\top\Sigmaw \) matches the variance of \( w^\topX \).
3. **Density contours** are ellipses aligned with the eigenvectors of \( \Sigma \).
4. **Conditionals** are again normal with the closed-form mean/variance from the PDF.
5. **Linear combinations** \( a^\topX \) are univariate normal — the defining property.

Next up: the conditional expectation viewpoint, which generalizes Part 4.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Sample $N=10^6$ draws from $N_2(\mu,\Sigma)$ with $\mu=(2,-1)^\top$ and $\Sigma=\begin{pmatrix}4 & 1\\ 1 & 9\end{pmatrix}$ using the Cholesky factor. Report the sample mean, the sample covariance, and $\max|\Sigma-\widehat\Sigma|$.

2. Let $R=(R_1,R_2,R_3)^\top\sim N_3(\mu,\Sigma)$ with $\mu=(0.05,0.08,0.04)^\top$ and $\Sigma$ as in Part 2. Find the minimum-variance portfolio $w\propto\Sigma^{-1}1$, normalize it to sum to 1, and report the resulting $w^\top\Sigmaw$. Confirm by simulation that $\operatorname{Var}(w^\topR)$ matches.

3. For the bivariate normal in Part 4 ($\mu_1=0,\mu_2=1,\sigma_1=1.5,\sigma_2=1,\rho=0.7$), plot the conditional mean $E[X_2\mid X_1=x]$ as a function of $x$ and overlay it on a scatter of samples. Report the slope and intercept and confirm they match $\rho\sigma_2/\sigma_1$ and $\mu_2-\rho\sigma_2\mu_1/\sigma_1$.

4. Take $X\sim N_3(\mu,\Sigma)$ from Part 5 and the vector $a=(0.7,-1.2,0.4)^\top$. State the exact distribution of $Y=a^\topX$ and run a KS test that the standardized samples of $Y$ are standard normal. Report the KS statistic and p-value.

5. Show that any sub-vector of an MVN is MVN: sample $X\sim N_3(\mu,\Sigma)$ as above, drop the middle coordinate, and verify that the remaining two-component vector has the mean and covariance predicted by deleting the corresponding row/column of $\mu$ and $\Sigma$.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Use `L = np.linalg.cholesky(Sigma)` and `X = mu + Z @ L.T`. With \(N=10^6\) and seed 42 the sample mean is approximately \((2.00,-1.00)^\top\), the sample covariance is within ~\(0.01\) of \(\Sigma\), and \(\max|\Sigma-\widehat\Sigma|\approx 0.015\).
```python
mu = np.array([2.0,-1.0]); Sig = np.array([[4,1],[1,9]])
L = np.linalg.cholesky(Sig); rng = np.random.default_rng(42)
Z = rng.standard_normal((1_000_000,2)); X = mu + Z @ L.T
print(X.mean(axis=0)); print(np.cov(X,rowvar=False,ddof=1))
print(np.max(np.abs(Sig - np.cov(X,rowvar=False,ddof=1))))
```

**2.** Minimum-variance weights: \(w\propto\Sigma^{-1}1\). Code:
```python
import numpy as np
mu = np.array([0.05,0.08,0.04])
Sig = np.array([[0.10,0.02,0.01],[0.02,0.16,-0.03],[0.01,-0.03,0.09]])
w = np.linalg.solve(Sig, np.ones(3)); w = w / w.sum()
print("w =", w, " w^T Sigma w =", w @ Sig @ w)
```
The simulation of \(w^\topR\) gives a sample variance matching \(w^\top\Sigmaw\) to within Monte Carlo error.

**3.** Slope \(=\rho\sigma_2/\sigma_1 = 0.7\cdot 1/1.5 \approx 0.4667\); intercept \(=\mu_2-\rho\sigma_2\mu_1/\sigma_1 = 1\). The scatter with the conditional mean line overlay should show the data cloud tilted along this line; binning samples in a window around any \(x\) reproduces the conditional mean to two decimals.

**4.** \(Y\sim N(a^\top\mu,\ a^\top\Sigmaa)\). With the numbers from Part 5, \(a^\top\mu\approx -2.3\) and \(a^\top\Sigmaa\approx 3.7\). Standardize \((Y-\text{mean})/\sqrt{\text{var}}\) and run `stats.kstest(..., "norm")`; the p-value should be comfortably above 0.05, so we fail to reject normality.

**5.** Dropping the middle coordinate gives a bivariate vector \((X_1,X_3)^\top\) whose mean is \((\mu_1,\mu_3)^\top\) and covariance is the \(2\times 2\) submatrix of \(\Sigma\) obtained by deleting row/column 2. The sample mean/covariance of the corresponding columns of `X` reproduce these to Monte Carlo error, confirming the sub-vector property.


</details>
